# 정규표현식 사용

In [2]:
import re
text = "주문번호: 12345 금액: 89,000원 이메일: test@test.com"
emails = re.findall(r"\w+@\w+\.\w+", text)
numbers = re.findall(r"\d+", text)
price = re.findall(r"\d{1,3},\d{3}", text)
print('Emails:', emails)
print('Numbers:', numbers)
print('Price:', price)

Emails: ['test@test.com']
Numbers: ['12345', '89', '000']
Price: ['89,000']


# IMAP 메일 가져오기

In [ ]:
import imaplib
mail = imaplib.IMAP4_SSL('imap.gmail.com')
mail.login('이메일','비밀전호')
mail.select('inbox')
print('Connected to mailbox')

Connected to mailbox


# 메일 본문 데이터 처리

In [4]:
body = "주문번호: 98765 금액: 120,000원"
order_id = re.findall(r"\d{5}", body)
price = re.findall(r"\d{1,3},\d{3}", body)
print(order_id, price)

['98765'] ['120,000']


# CSV 저장 

In [5]:
import csv
data = [['order_id', 'price'],['98765', '120000']]
with open('result.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(data)
print('CSV 저장 완료')

CSV 저장 완료


# 메일에서 첨부파일 내려받아 텍스트 추출하기

In [ ]:
import imaplib
import email
from email.header import decode_header
import os
import pdfplumber
IMAP_SERVER = "imap.gmail.com"
EMAIL = '이메일'
PASSWORD = '비밀번호'
SAVE_DIR = "./downloads"
os.makedirs(SAVE_DIR, exist_ok=True)
# 1. IMAP 접속
mail = imaplib.IMAP4_SSL(IMAP_SERVER)
mail.login(EMAIL, PASSWORD)
mail.select("inbox")

('OK', [b'625'])

In [8]:
# 2. 메일 검색 (제목 필터는 직접 처리)
result, data = mail.search(None, "ALL")
mail_ids = data[0].split()
target_mail = None
# 최근 메일부터 역순 탐색
for mail_id in reversed(mail_ids):
    result, data = mail.fetch(mail_id, "(RFC822)")
    raw_email = data[0][1]
    msg = email.message_from_bytes(raw_email)
    # 제목 디코딩
    subject, encoding = decode_header(msg["Subject"])[0]
    if isinstance(subject, bytes):
        subject = subject.decode(encoding or "utf-8")
    if "[업무협조]" in subject:
        print("대상 메일:", subject)
        target_mail = msg
        break
    
if target_mail is None:
    print(" 조건에 맞는 메일 없음")
# exit()

대상 메일: [업무협조] 기능 검토 요청


In [9]:
# 3. PDF 첨부파일 다운로드
pdf_files = []
for part in target_mail.walk():
    content_disposition = part.get("Content-Disposition")
    if content_disposition and "attachment" in content_disposition:
        filename = part.get_filename()

        if filename:
            filename, enc = decode_header(filename)[0]
            if isinstance(filename, bytes):
                filename = filename.decode(enc or "utf-8")
            if filename.lower().endswith(".pdf"):
                filepath = os.path.join(SAVE_DIR, filename)
                with open(filepath, "wb") as f:
                    f.write(part.get_payload(decode=True))
                print(f"다운로드 완료: {filepath}")
                pdf_files.append(filepath)

다운로드 완료: ./downloads\mail1_request.pdf


In [10]:
# 4. PDF 텍스트 추출
for pdf_path in pdf_files:
    print(f"\n텍스트 추출: {pdf_path}")
    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
print("---- 추출 텍스트 ----")
print(full_text[:1000]) # 일부만 출력


텍스트 추출: ./downloads\mail1_request.pdf
---- 추출 텍스트 ----
[nnn nnnn nn nnn]
nnnnn nn nn nn nnn nnnnnnn.
nn nnn nnn nnn nn nn nnn nnnnn.
nn nn:
- nn nn nn nn nn
- nnn nnn nn
- nn nn nn
nn nn: 2026-05-03

